# SMILES-2026 — Colab runner (real Qwen2.5-0.5B)

This notebook **does not change** your challenge code (`solution.py`, `aggregation.py`, `probe.py`, `splitting.py`). It only installs dependencies and runs `python solution.py` on a GPU runtime.

**Runtime:** `Runtime` → `Change runtime type` → **GPU** (T4 is enough).

**Optional:** put a [Hugging Face token](https://huggingface.co/settings/tokens) in Colab secrets as `HF_TOKEN` if you hit rate limits (see cell below).


In [ ]:
# GPU check
!nvidia-smi
import torch
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## 1) Get the repository into Colab

Pick **one** path:

- **A) Public Git repo** — set `REPO_URL` and run the clone cell.
- **B) Upload** — zip your project from your PC, use the sidebar **folder** icon → **Upload**, unzip, then `cd` into it in the next cell.


In [ ]:
# --- Option A: clone (edit REPO_URL) ---
import os
import subprocess

REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # <-- change this
CLONE_DIR = "/content/SMILES-2026-Hallucination-Detection"

if not os.path.isfile(os.path.join(CLONE_DIR, "solution.py")):
    subprocess.run(["git", "clone", REPO_URL, CLONE_DIR], check=True)
else:
    print("Repo folder already exists; skipping clone.")

os.chdir(CLONE_DIR)
print("cwd:", os.getcwd())

In [ ]:
# If you used Option B (upload zip), uncomment and set the folder name:
# %cd /content/YOUR_UNZIPPED_FOLDER

import os
assert os.path.isfile("solution.py"), "Run from repo root (solution.py missing)."

In [ ]:
# Dependencies (matches requirements.txt)
!pip install -q torch transformers datasets scikit-learn numpy pandas tqdm

# Optional: HF token from Colab secrets (recommended if downloads fail)
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok:
        import os
        os.environ["HF_TOKEN"] = tok
        os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
        print("HF_TOKEN loaded from Colab secrets.")
except Exception:
    print("No Colab secrets / HF_TOKEN — anonymous downloads only.")

In [ ]:
# Real Qwen run: do NOT enable the local CPU stub
import os
os.environ.pop("SMILES_STUB_LM", None)
os.environ["PYTHONIOENCODING"] = "utf-8"

print("SMILES_STUB_LM:", os.environ.get("SMILES_STUB_LM", "(unset — correct for real model)"))

In [ ]:
# Run the official entrypoint (~15–40 min on T4 typical; first run downloads model)
!python solution.py

In [ ]:
# Summarize outputs
import json
from pathlib import Path

p = Path("results.json")
if p.exists():
    r = json.loads(p.read_text(encoding="utf-8"))
    print("feature_dim:", r.get("feature_dim"))
    print("extract_time_s:", r.get("extract_time_s"))
    print("avg_test_accuracy:", r.get("avg_test_accuracy"))
    print("avg_test_f1:", r.get("avg_test_f1"))
    print("avg_test_auroc:", r.get("avg_test_auroc"))
else:
    print("results.json not found")

pred = Path("predictions.csv")
if pred.exists():
    print("predictions.csv lines:", sum(1 for _ in pred.open(encoding="utf-8")))

In [ ]:
# Download artifacts to your computer
try:
    from google.colab import files
    files.download("results.json")
    files.download("predictions.csv")
except ImportError:
    print("Not on Colab — files are already in the working directory.")